## Install dependencies

In [35]:
!pip install -q -U sentence-transformers google-generativeai groq tqdm


## Imports & paths

In [ ]:
import os, re, json, random, time
from pathlib import Path
import numpy as np
from collections import defaultdict, Counter

random.seed(42)

DATASET_PATH    = "/kaggle/input/datasets/quiz_dataset.json"
OUTPUT_DIR      = "/kaggle/working"

EVAL_RESULTS_PATH  = f"{OUTPUT_DIR}/eval_results.json"
EVAL_PROGRESS_PATH = f"{OUTPUT_DIR}/eval_progress.json"
REPORT_PATH        = f"{OUTPUT_DIR}/eval_report.json"

GEMINI_API_KEYS = [
    "",
    "",
]

GROQ_API_KEYS = [
    "",
    "",
]

JUDGE_STRATEGY = "gemini_first"

SAMPLE_PER_COURSE_PER_MODE = 5
assert os.path.exists(DATASET_PATH), f"Dataset not found at '{DATASET_PATH}'. Update DATASET_PATH."
print("Paths OK ")
print(f"Eval results  → {EVAL_RESULTS_PATH}")
print(f"Report        → {REPORT_PATH}")
print(f"Judge strategy: {JUDGE_STRATEGY}")
print(f"Gemini keys   : {len(GEMINI_API_KEYS)}")
print(f"Groq keys     : {len(GROQ_API_KEYS)}")


Paths OK ✅
Eval results  → /kaggle/working/eval_results.json
Report        → /kaggle/working/eval_report.json
Judge strategy: groq_only
Gemini keys   : 2
Groq keys     : 2


## Load dataset & build stratified sample

In [37]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Total samples loaded: {len(dataset)}")

groups = defaultdict(list)
for i, item in enumerate(dataset):
    groups[(item["course"], item["mode"])].append(i)

print(f"Unique (course, mode) groups: {len(groups)}")

sample_indices = []
for (course, mode), idxs in sorted(groups.items()):
    k = min(SAMPLE_PER_COURSE_PER_MODE, len(idxs))
    sample_indices.extend(random.sample(idxs, k))

sample_indices = sorted(set(sample_indices))
eval_samples   = [dataset[i] for i in sample_indices]

print(f"Evaluation sample size: {len(eval_samples)} samples")
print(f"  ({SAMPLE_PER_COURSE_PER_MODE} per course×mode, stratified across {len(set(d['course'] for d in eval_samples))} courses × 3 modes)")
print("Sample ready")

Total samples loaded: 8042
Unique (course, mode) groups: 60
Evaluation sample size: 300 samples
  (5 per course×mode, stratified across 20 courses × 3 modes)
Sample ready ✅


## Load sentence-transformers embedder

In [38]:
import torch
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print("Embedder loaded ")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder loaded ✅
Device: cuda


## All shared helpers

In [39]:
import google.generativeai as genai
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

_gemini_key_idx = 0

def _get_gemini_model():
    global _gemini_key_idx
    key = GEMINI_API_KEYS[_gemini_key_idx % len(GEMINI_API_KEYS)]
    genai.configure(api_key=key)
    return genai.GenerativeModel(
        "gemini-2.0-flash-lite",
        generation_config=genai.GenerationConfig(
            temperature=0.0,
            max_output_tokens=256,
        )
    )

def _gemini_call(prompt, max_retries=3):
    """Try every Gemini key once before giving up. Returns text or None."""
    global _gemini_key_idx
    wait = 20
    keys_tried = 0
    for attempt in range(max_retries * len(GEMINI_API_KEYS)):
        try:
            model = _get_gemini_model()
            resp  = model.generate_content(prompt)
            return resp.text.strip()
        except Exception as e:
            err = str(e).lower()
            if "quota" in err or "rate" in err or "429" in err or "resource" in err:
                _gemini_key_idx += 1
                keys_tried += 1
                print(f"  [gemini rate-limit] rotating to key {_gemini_key_idx % len(GEMINI_API_KEYS)}, waiting {wait}s…")
                time.sleep(wait)
                wait = min(wait * 1.5, 120)
                if keys_tried >= len(GEMINI_API_KEYS):
                    print("  [gemini] all keys rate-limited — giving up on Gemini for this call")
                    return None
            elif "token" in err or "budget" in err:
                print(f"  [gemini token-budget] waiting 60s…")
                time.sleep(60)
            else:
                print(f"  [gemini error] {e}")
                time.sleep(10)
    return None

_groq_key_idx = 0

def _get_groq_client():
    global _groq_key_idx
    from groq import Groq
    return Groq(api_key=GROQ_API_KEYS[_groq_key_idx % len(GROQ_API_KEYS)])

def _groq_call(prompt, max_retries=4):
    """Call Groq with key rotation on rate-limit. Returns text or None."""
    global _groq_key_idx
    wait = 10
    for attempt in range(max_retries * len(GROQ_API_KEYS)):
        try:
            client = _get_groq_client()
            resp = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=256,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err = str(e).lower()
            if "rate" in err or "429" in err or "limit" in err:
                _groq_key_idx += 1
                print(f"  [groq rate-limit] rotating to key {_groq_key_idx % len(GROQ_API_KEYS)}, waiting {wait}s…")
                time.sleep(wait)
                wait = min(wait * 2, 60)
            else:
                print(f"  [groq error] {e}")
                time.sleep(5)
    return None


def judge_api_call(prompt):
    """
    Routes the prompt based on JUDGE_STRATEGY.
    groq_only    → Groq with key rotation, no fallback
    groq_first   → Groq first, Gemini if all Groq keys fail
    gemini_first → Gemini first, Groq if all Gemini keys fail
    """
    if JUDGE_STRATEGY == "groq_only":
        result = _groq_call(prompt)
        if result is None:
            print("  [judge] all Groq keys exhausted — skipping sample")
        return result

    if JUDGE_STRATEGY == "groq_first":
        result = _groq_call(prompt)
        if result is not None:
            return result
        print("  [judge] Groq exhausted → falling back to Gemini…")
        return _gemini_call(prompt)

    # gemini_first
    result = _gemini_call(prompt)
    if result is not None:
        return result
    print("  [judge] Gemini exhausted → falling back to Groq…")
    return _groq_call(prompt)


def extract_questions_text(output: str) -> str:
    """Strip answer lines, return only question stems + options for embedding."""
    lines = []
    for line in output.split("\n"):
        if re.match(r"^Answer:\s*", line, re.IGNORECASE):
            continue
        lines.append(line)
    return " ".join(lines).strip()


def compute_semantic_similarity(chunk: str, output: str) -> float:
    """Cosine similarity between chunk embedding and question text embedding."""
    q_text = extract_questions_text(output)
    embs   = embedder.encode([chunk, q_text], normalize_embeddings=True)
    return float(np.dot(embs[0], embs[1]))


_JUDGE_PROMPT_TEMPLATE = """You are an expert evaluator for university quiz question quality.
You will be given a lecture text chunk and a set of generated quiz questions with their answers.
Evaluate the questions on THREE dimensions and return ONLY a JSON object — no preamble, no explanation.

DIMENSIONS:
1. faithfulness   (1-5): Are all questions and answers fully supported by the lecture text?
                          5 = every claim is in the text. 1 = questions contain hallucinated facts.
2. correctness    (1-5): Are the marked answers actually correct given the lecture text?
                          5 = all answers correct. 1 = most answers wrong.
3. question_quality (1-5): Are questions well-formed, non-trivial, and suitable for a university exam?
                          5 = excellent exam questions. 1 = trivial, vague, or copied verbatim.

Also add a brief "issues" field (string) noting any specific problems, or "none" if all looks good.

Return ONLY this JSON (no markdown, no backticks):
{{"faithfulness": <1-5>, "correctness": <1-5>, "question_quality": <1-5>, "issues": "<text>"}}

---
LECTURE TEXT:
{chunk}

---
GENERATED QUESTIONS:
{output}
"""

def judge_sample(chunk: str, output: str) -> dict | None:
    """Call the judge. Returns dict with scores or None on failure."""
    prompt = _JUDGE_PROMPT_TEMPLATE.format(
        chunk=chunk[:1200],
        output=output[:1000],
    )
    raw = judge_api_call(prompt)
    if raw is None:
        return None
    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        return json.loads(clean)
    except json.JSONDecodeError:
        m = re.search(r'\{.*?\}', raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except:
                pass
        print(f"  [parse error] could not parse: {raw[:200]}")
        return None


def load_eval_progress():
    if os.path.exists(EVAL_RESULTS_PATH) and os.path.exists(EVAL_PROGRESS_PATH):
        with open(EVAL_RESULTS_PATH, "r", encoding="utf-8") as f:
            results = json.load(f)
        with open(EVAL_PROGRESS_PATH, "r", encoding="utf-8") as f:
            progress = json.load(f)
        done_keys = set(progress["done_keys"])
        print(f"Resuming: {len(results)} results, {len(done_keys)} samples done.")
        return results, done_keys
    print("No prior progress — starting fresh.")
    return [], set()

def save_eval_progress(results, done_keys):
    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    with open(EVAL_PROGRESS_PATH, "w", encoding="utf-8") as f:
        json.dump({"done_keys": list(done_keys)}, f, indent=2)


print("Shared helpers ready ")
print(f"Judge strategy : {JUDGE_STRATEGY}")
print(f"Groq keys      : {len(GROQ_API_KEYS)} (llama-3.1-8b-instant)")
print(f"Gemini keys    : {len(GEMINI_API_KEYS)}")


Shared helpers ready ✅
Judge strategy : groq_only
Groq keys      : 2 (llama-3.1-8b-instant)
Gemini keys    : 2


## Semantic Similarity

In [40]:
from tqdm import tqdm

SIM_THRESHOLD = 0.40 

results, done_keys = load_eval_progress()

results_by_key = {r["key"]: r for r in results}

sim_todo = [
    item for item in eval_samples
    if f"{item['course']}||{item['chunk_index']}||{item['mode']}" not in results_by_key
    or "semantic_similarity" not in results_by_key[f"{item['course']}||{item['chunk_index']}||{item['mode']}"]
]
print(f"{len(sim_todo)} samples need similarity scores.")

for item in tqdm(sim_todo, desc="Semantic similarity"):
    key = f"{item['course']}||{item['chunk_index']}||{item['mode']}"
    sim = compute_semantic_similarity(item["input"], item["output"])
    flag = "LOW_SIMILARITY" if sim < SIM_THRESHOLD else "OK"

    if key in results_by_key:
        results_by_key[key]["semantic_similarity"] = round(sim, 4)
        results_by_key[key]["sim_flag"] = flag
    else:
        entry = {
            "key": key,
            "course": item["course"],
            "mode": item["mode"],
            "chunk_index": item["chunk_index"],
            "n_questions": item["n_questions"],
            "semantic_similarity": round(sim, 4),
            "sim_flag": flag,
            "faithfulness": None,
            "correctness": None,
            "question_quality": None,
            "issues": None,
            "judge_done": False,
        }
        results.append(entry)
        results_by_key[key] = entry

    done_keys.add(key + "||sim")
    save_eval_progress(results, done_keys)

sims = [r["semantic_similarity"] for r in results if r.get("semantic_similarity") is not None]
low  = sum(1 for s in sims if s < SIM_THRESHOLD)
print(f"\n Similarity pass complete")
print(f"   Mean similarity : {sum(sims)/len(sims):.4f}")
print(f"   Min / Max       : {min(sims):.4f} / {max(sims):.4f}")
print(f"   LOW_SIMILARITY  : {low}/{len(sims)} ({100*low/len(sims):.1f}%)")

Resuming: 300 results, 300 samples done.
0 samples need similarity scores.


Semantic similarity: 0it [00:00, ?it/s]


✅ Similarity pass complete
   Mean similarity : 0.6893
   Min / Max       : 0.2130 / 0.9171
   LOW_SIMILARITY  : 8/300 (2.7%)


## Layer 2 — LLM-as-Judge (Gemini + Groq fallback)

For each sample in the evaluation set, the judge scores:
- **Faithfulness** (1–5): questions grounded in the source chunk
- **Correctness** (1–5): marked answers are actually correct
- **Question Quality** (1–5): well-formed, non-trivial university exam questions


In [41]:
from tqdm import tqdm

sample_lookup = {
    f"{s['course']}||{s['chunk_index']}||{s['mode']}": s
    for s in eval_samples
}

judge_todo = [r for r in results if not r.get("judge_done", False)]
print(f"{len(judge_todo)} samples need judge scores.")
print(f"Strategy: {JUDGE_STRATEGY} | Groq keys: {len(GROQ_API_KEYS)}")


INTER_CALL_SLEEP = 3

failed_judge = 0

for r in tqdm(judge_todo, desc="LLM-as-Judge"):
    key  = r["key"]
    item = sample_lookup.get(key)
    if item is None:
        continue

    scores = judge_sample(item["input"], item["output"])

    if scores is None:
        failed_judge += 1
        r["judge_done"] = False
    else:
        r["faithfulness"]     = scores.get("faithfulness")
        r["correctness"]      = scores.get("correctness")
        r["question_quality"] = scores.get("question_quality")
        r["issues"]           = scores.get("issues", "none")
        r["judge_done"]       = True

    save_eval_progress(results, done_keys)
    time.sleep(INTER_CALL_SLEEP)

judged = [r for r in results if r.get("judge_done")]
print(f"\n Judge pass complete")
print(f"   Scored          : {len(judged)}/{len(results)}")
print(f"   Failed          : {failed_judge}")
if judged:
    for dim in ["faithfulness", "correctness", "question_quality"]:
        vals = [r[dim] for r in judged if r.get(dim) is not None]
        print(f"   Mean {dim:<20}: {sum(vals)/len(vals):.2f} / 5")


172 samples need judge scores.
Strategy: groq_only | Groq keys: 2


LLM-as-Judge: 100%|██████████| 172/172 [21:51<00:00,  7.63s/it]


✅ Judge pass complete
   Scored          : 300/300
   Failed          : 0
   Mean faithfulness        : 4.24 / 5
   Mean correctness         : 4.52 / 5
   Mean question_quality    : 3.94 / 5


## Generate Summary Report

In [42]:
with open(EVAL_RESULTS_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

judged = [r for r in results if r.get("judge_done")]
print(f"Scored samples : {len(judged)} / {len(results)}")

DIMS = ["faithfulness", "correctness", "question_quality"]


def avg(lst):
    lst = [x for x in lst if x is not None]
    return round(sum(lst) / len(lst), 3) if lst else None


def pct_above(lst, threshold=4):
    lst = [x for x in lst if x is not None]
    return round(100 * sum(1 for x in lst if x >= threshold) / len(lst), 1) if lst else None


overall = {
    "total_dataset_samples": len(dataset),
    "eval_sample_size": len(results),
    "judge_scored": len(judged),
    "semantic_similarity": {
        "mean": avg([r["semantic_similarity"] for r in results]),
        "pct_above_0_40": pct_above([r["semantic_similarity"] for r in results], 0.40),
        "low_similarity_count": sum(1 for r in results if r.get("sim_flag") == "LOW_SIMILARITY"),
    },
}
for dim in DIMS:
    vals = [r[dim] for r in judged]
    overall[dim] = {
        "mean": avg(vals),
        "pct_above_4": pct_above(vals, 4),
    }

by_course = defaultdict(list)
for r in judged:
    by_course[r["course"]].append(r)

course_report = {}
for course, rows in sorted(by_course.items()):
    entry = {"n": len(rows)}
    entry["semantic_similarity_mean"] = avg([r["semantic_similarity"] for r in rows])
    for dim in DIMS:
        entry[f"{dim}_mean"] = avg([r[dim] for r in rows])
        entry[f"{dim}_pct_above_4"] = pct_above([r[dim] for r in rows], 4)
    course_report[course] = entry

by_mode = defaultdict(list)
for r in judged:
    by_mode[r["mode"]].append(r)

mode_report = {}
for mode, rows in sorted(by_mode.items()):
    entry = {"n": len(rows)}
    entry["semantic_similarity_mean"] = avg([r["semantic_similarity"] for r in rows])
    for dim in DIMS:
        entry[f"{dim}_mean"] = avg([r[dim] for r in rows])
        entry[f"{dim}_pct_above_4"] = pct_above([r[dim] for r in rows], 4)
    mode_report[mode] = entry

issues_samples = [
    {"key": r["key"], "issues": r["issues"]}
    for r in judged
    if r.get("issues") and r["issues"].lower() not in ("none", "no issues", "")
]

report = {
    "overall": overall,
    "by_course": course_report,
    "by_mode": mode_report,
    "flagged_issues": issues_samples[:50],
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\nReport saved → {REPORT_PATH}")

print("\n" + "═"*60)
print("  OVERALL EVALUATION SUMMARY")
print("═"*60)
print(f"  Dataset size        : {overall['total_dataset_samples']:,} samples")
print(f"  Eval sample         : {overall['eval_sample_size']} samples (stratified)")
print(f"  Judge scored        : {overall['judge_scored']}")
print()
sim = overall["semantic_similarity"]
print(f"  Semantic Similarity : mean={sim['mean']:.4f}  |  {sim['pct_above_0_40']}% ≥ 0.40")
print(f"  Low-similarity flags: {sim['low_similarity_count']}")
print()
for dim in DIMS:
    d = overall[dim]
    print(f"  {dim:<20}: mean={d['mean']:.2f}/5  |  {d['pct_above_4']}% scored ≥4")
print()
print("─"*60)
print("  BY MODE")
print("─"*60)
for mode, d in mode_report.items():
    print(f"  {mode:<8} (n={d['n']:>3})  faith={d['faithfulness_mean']:.2f}  correct={d['correctness_mean']:.2f}  quality={d['question_quality_mean']:.2f}")
print()
print("─"*60)
print("  BY COURSE (top 5 by faithfulness)")
print("─"*60)
sorted_courses = sorted(course_report.items(), key=lambda x: x[1].get("faithfulness_mean") or 0, reverse=True)
for course, d in sorted_courses[:5]:
    print(f"  {course:<35} faith={d['faithfulness_mean']:.2f}  correct={d['correctness_mean']:.2f}")
print("  ...")
print()
print(f"  Samples with noted issues: {len(issues_samples)}")
print("═"*60)

Scored samples : 300 / 300

Report saved → /kaggle/working/eval_report.json

════════════════════════════════════════════════════════════
  OVERALL EVALUATION SUMMARY
════════════════════════════════════════════════════════════
  Dataset size        : 8,042 samples
  Eval sample         : 300 samples (stratified)
  Judge scored        : 300

  Semantic Similarity : mean=0.6890  |  97.3% ≥ 0.40
  Low-similarity flags: 8

  faithfulness        : mean=4.24/5  |  92.3% scored ≥4
  correctness         : mean=4.52/5  |  93.7% scored ≥4
  question_quality    : mean=3.94/5  |  83.3% scored ≥4

────────────────────────────────────────────────────────────
  BY MODE
────────────────────────────────────────────────────────────
  mcq      (n=100)  faith=4.49  correct=4.74  quality=4.23
  mixed    (n=100)  faith=4.25  correct=4.54  quality=3.88
  tf       (n=100)  faith=3.99  correct=4.29  quality=3.70

────────────────────────────────────────────────────────────
  BY COURSE (top 5 by faithfulness)


In [43]:
with open(EVAL_RESULTS_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

judged = [r for r in results if r.get("judge_done")]

for r in judged:
    scores = [r.get(d) for d in ["faithfulness", "correctness", "question_quality"] if r.get(d) is not None]
    r["composite"] = round(sum(scores) / len(scores), 2) if scores else 0

worst = sorted(judged, key=lambda x: x["composite"])[:10]

print("10 LOWEST-SCORING SAMPLES")
print("═"*70)
for i, r in enumerate(worst, 1):
    print(f"\n[{i}] {r['course']} | chunk {r['chunk_index']} | {r['mode']}")
    print(f"    Composite={r['composite']}  Faith={r.get('faithfulness')}  Correct={r.get('correctness')}  Quality={r.get('question_quality')}")
    print(f"    Semantic sim : {r.get('semantic_similarity')}")
    print(f"    Issues       : {r.get('issues')}")

    key  = r["key"]
    item = sample_lookup.get(key)
    if item:
        print(f"\n    --- CHUNK (first 400 chars) ---")
        print(f"    {item['input'][:400]}")
        print(f"\n    --- OUTPUT ---")
        print(f"    {item['output'][:600]}")
    print("─"*70)

10 LOWEST-SCORING SAMPLES
══════════════════════════════════════════════════════════════════════

[1] Computational_Geometry | chunk 2 | tf
    Composite=1.33  Faith=1  Correct=1  Quality=2
    Semantic sim : 0.3713
    Issues       : questions contain claims not supported by the lecture text, and the text does not provide enough information to verify the answers

    --- CHUNK (first 400 chars) ---
    Books Orientation Tests Orientation Tests Orientation Tests Orientation Tests Orientation Tests Orientation Tests Orientation Tests Orientation Tests Simple Polygon and Convex Polygon CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL CONVEX HULL 

    --- OUTPUT ---
    Q1: The convex hull can be computed using the brute-force algorithm which checks all possible subsets of points to find the smallest subset forming a convex polygon.
An

## Final dataset stats

In [44]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

courses   = Counter(d["course"]      for d in dataset)
modes     = Counter(d["mode"]        for d in dataset)
n_qs      = Counter(d["n_questions"] for d in dataset)

print(f"{'─'*55}")
print(f"  FULL DATASET STATISTICS")
print(f"{'─'*55}")
print(f"  Total samples : {len(dataset):,}")
print()
print(f"  By mode:")
for m, c in sorted(modes.items()):
    print(f"    {m:<10} {c:>5}  ({100*c/len(dataset):.1f}%)")
print()
print(f"  By n_questions:")
for n, c in sorted(n_qs.items()):
    print(f"    {n} questions  {c:>5}  ({100*c/len(dataset):.1f}%)")
print()
print(f"  By course ({len(courses)} total):")
for course, c in sorted(courses.items(), key=lambda x: -x[1]):
    print(f"    {course:<38} {c:>4}")
print(f"{'─'*55}")

───────────────────────────────────────────────────────
  FULL DATASET STATISTICS
───────────────────────────────────────────────────────
  Total samples : 8,042

  By mode:
    mcq         2687  (33.4%)
    mixed       2671  (33.2%)
    tf          2684  (33.4%)

  By n_questions:
    2 questions   2948  (36.7%)
    5 questions   2337  (29.1%)
    8 questions   2757  (34.3%)

  By course (20 total):
    GIS                                     717
    Cloud_Computing                         696
    Software_Quality_Assurance              693
    Compilers_Theory                        555
    neural_networks                         520
    HCI                                     519
    cyber_security                          464
    data_science                            451
    LP                                      436
    NLP                                     406
    Statistical_Inference                   360
    Distributed                             351
    Social_Network_A